In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import folium




In [ ]:
ee.Authenticate()
ee.Initialize(project = 'smart-oasis-466614-s5')

In [ ]:
pouso_alegre = ee.Geometry.Point([-45.948889, -22.230278])

In [ ]:
startDate = '2020-01-01'
endDate = '2025-05-31'

In [ ]:
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterDate(startDate, endDate)
                  .filterBounds(pouso_alegre)
                  # Filtro para pegar imagens com menos de 20% de nuvens
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

In [ ]:
def add_ndvi(image):
  # NDVI = (NIR - Red) / (NIR + Red) -> Para Sentinel-2: (B8 - B4) / (B8 + B4)
  ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
  # Multiplicamos por 10000 para manter a escala original e salvamos como inteiro
  # Isso ajuda a evitar problemas de precisão e armazenamento.
  return image.addBands(ndvi.multiply(10000).toInt16())

# Aplicar a função à coleção para obter o NDVI
ndvi_collection = s2_collection.map(add_ndvi)

# --- ETAPA 3: Função Ajustada para Extrair a Série Temporal ---
def create_point_time_series_df(image_collection, geometry, band_name, scale=30):
  """
  Extrai a série temporal para um ponto, usando a média dos pixels ao redor.
  """
  def get_value_at_point(image):
    # Usamos .mean() para obter a média dos pixels em uma área definida pela escala
    # É mais robusto do que pegar o valor de um único pixel.
    reduced_value = image.reduceRegion(
        reducer=ee.Reducer.mean(), # MUDANÇA PRINCIPAL: de .sum() para .mean()
        geometry=geometry,
        scale=scale,              # MUDANÇA PRINCIPAL: escala apropriada para Sentinel-2
        maxPixels=1e9
    )
    # Define a propriedade na imagem com o valor extraído
    return image.set(reduced_value)

  # Mapeia a função sobre a coleção
  reduced_collection = image_collection.map(get_value_at_point)

  # Extrai os valores e as datas. O filtro remove imagens onde não foi possível extrair valor.
  time_series_values = reduced_collection.filter(ee.Filter.notNull([band_name])).aggregate_array(band_name).getInfo()
  time_series_time = reduced_collection.filter(ee.Filter.notNull([band_name])).aggregate_array('system:time_start').getInfo()

  # Cria o DataFrame com pandas
  df = pd.DataFrame({'time_start': time_series_time, band_name: time_series_values})
  df['time_start'] = pd.to_datetime(df['time_start'], unit='ms')
  df.set_index('time_start', inplace=True)

  # O valor original do NDVI vai de -1 a 1. Dividimos por 10000 para reverter a multiplicação anterior.
  df[band_name] = df[band_name] / 10000

  return df

# --- ETAPA 4: Executar a Função e Visualizar o Resultado ---
# Chame a função com a coleção NDVI, a geometria e o nome da banda
ndvi_time_series_df = create_point_time_series_df(ndvi_collection, pouso_alegre, 'NDVI')

# Mostrar as primeiras linhas do DataFrame resultante
print(ndvi_time_series_df.head())


                           NDVI
time_start                     
2020-01-07 13:18:21.985  0.7612
2020-01-07 13:18:18.886  0.7508
2020-01-27 13:18:20.472  0.7831
2020-01-27 13:18:17.375  0.7814
2020-03-07 13:18:23.474  0.7798


In [ ]:
import folium
def add_ee_layer(self, ee_image_object, vis_params, name):
  """
  Adiciona uma camada de imagem do Google Earth Engine (ee.Image) a um mapa Folium.
  """
  map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
  folium.raster_layers.TileLayer(
      tiles=map_id_dict['tile_fetcher'].url_format,
      attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
      name=name,
      overlay=True,
      control=True
  ).add_to(self)
folium.Map.add_ee_layer = add_ee_layer

In [ ]:
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterDate(startDate, endDate)
                 .filterBounds(pouso_alegre)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

def add_ndvi(image):
  ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
  return image.addBands(ndvi)

ndvi_collection = s2_collection.map(add_ndvi)

# --- ETAPA 4: Preparar a Imagem para o Mapa ---
median_ndvi_image = ndvi_collection.select('NDVI').median()

# --- ETAPA 5: Definir Parâmetros de Visualização ---
ndvi_vis_params = {
  'min': -0.2,
  'max': 0.9,
  'palette': ['red', 'orange', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# --- ETAPA 6: Visualizar no Mapa ---
# Obter coordenadas para centrar o mapa
center_coords = pouso_alegre.coordinates().getInfo()
map_center = [center_coords[1], center_coords[0]] # Lat, Lon

# Crie um mapa Folium
my_map = folium.Map(location=map_center, zoom_start=11)

# AGORA ESTA LINHA VAI FUNCIONAR!
my_map.add_ee_layer(median_ndvi_image, ndvi_vis_params, 'NDVI Mediano')

# Adicione a geometria do ponto de Pouso Alegre (opcional)
my_map.add_child(folium.Marker(location=map_center, popup="Pouso Alegre"))

# Adicione o polígono da área de interesse (opcional)
my_map.add_child(folium.GeoJson(pouso_alegre.getInfo(), name='Área de Interesse'))

# Adicione controle de camadas para ligar/desligar
my_map.add_child(folium.LayerControl())

# Exibir o mapa
my_map